# Sensor Datasheet Quiz: Frame Rate, Image Size, and Storage

**Goal:** given a real camera sensor datasheet, a memory budget, and a communication bandwidth budget, calculate how fast you can stream images, how large each image is, and how long you can record.

This notebook follows directly from the *Practical Topics in Spatial AI* lecture on sensor architecture and the imaging pipeline.

By the end, you should be able to:

1. Read a sensor datasheet and extract the relevant specifications.
2. Calculate the raw image size in bytes from resolution and bit depth.
3. Determine the maximum frame rate that fits within a given communication bandwidth.
4. Determine the maximum recording time that fits within a given memory capacity.

---

## What you need

- Python 3
- NumPy
- Matplotlib

No hardware required.

# 1. Background

## 1.1 The sensor architecture pipeline

Every camera sensor follows this pipeline:

```text
Sensing Unit → Processing Unit → Transmission Unit
     ↓               ↓                  ↓
  Pixels        Memory / ISP        Interface
  (ADC)         (frame buffer)      (USB, GigE, ...)
```

In a real robotic system, two resources constrain how much data you can handle:

| Resource | Constraint |
|---|---|
| **Communication bandwidth** | limits how many frames per second you can *transmit* |
| **Memory capacity** | limits how long you can *store* images |

## 1.2 Image size in bytes

A raw image without compression:

$$
\text{Image size} = N_u \times N_v \times \text{bit depth} / 8 \quad [\text{bytes}]
$$

For a colour (Bayer/RGB) image:

$$
\text{Image size} = N_u \times N_v \times \text{bit depth} \times n_{\text{channels}} / 8 \quad [\text{bytes}]
$$

## 1.3 Maximum frame rate from bandwidth

If your link bandwidth is $B$ bits/s and one image is $S$ bytes:

$$
f_{\text{max}} = \frac{B}{S \times 8} \quad [\text{fps}]
$$

## 1.4 Maximum recording time from memory

If your memory capacity is $M$ bytes and each image is $S$ bytes, recording at $f$ fps:

$$
t_{\text{max}} = \frac{M}{S \times f} \quad [\text{seconds}]
$$

# 2. The quiz scenario

You are setting up the vision system for the MIRTE robot. You have been given:

| Resource | Value |
|---|---|
| Memory capacity | **10 GB** |
| Communication bandwidth | **1 Gbps** (gigabit per second) |

You will work with the **Allied Vision Manta G-235C** camera. The datasheet excerpt is in the next section — read through it and identify the values you need before writing any code.

You will answer two questions:

1. What is the **maximum frame rate** you can transmit over the 1 Gbps link for each pixel format?
2. What is the **maximum recording time** you can achieve with 10 GB of storage for each pixel format?

> **Tip:** work through the numbers on paper first. The code cells below will let you verify and explore.

## Allied Vision Manta G-235C — Datasheet Reference

The table below is an excerpt from the official product datasheet. Not every row is needed for the exercises — part of the task is identifying which values are relevant and noting the units carefully.

| Parameter | Value |
|---|---|
| **Sensor type** | CMOS, global shutter |
| **Resolution** | 1936 (H) × 1216 (V) — 2.35 MP |
| **Pixel size** | 5.86 µm × 5.86 µm |
| **Max frame rate (full resolution)** | 50.8 fps |
| **ADC bit depth** | 12-bit (max) |
| **Pixel formats — monochrome** | Mono8, Mono12 |
| **Pixel formats — colour** | BayerRG8, BayerRG12, RGB8Packed |
| **Interface** | GigE Vision (1 Gbps) |
| **Lens mount** | C-mount |


# 3. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)

# 4. System parameters

Run this cell as-is. These are the given values.

In [ ]:
# --- System resources ---
memory_GB        = 10.0          # GB
bandwidth_Gbps   = 1.0           # Gbps (gigabits per second)

# Convert to base units
memory_bytes     = memory_GB * 1e9          # bytes  (using 1 GB = 10^9 bytes)
bandwidth_bps    = bandwidth_Gbps * 1e9     # bits per second

print(f"Memory:    {memory_GB} GB  = {memory_bytes:.2e} bytes")
print(f"Bandwidth: {bandwidth_Gbps} Gbps = {bandwidth_bps:.2e} bps")

# --- Sensor parameters ---
N_u              = 1936          # horizontal pixels
N_v              = 1216          # vertical pixels
max_sensor_fps   = 50.8          # maximum frame rate of the sensor hardware

total_pixels = N_u * N_v
print(f"\nSensor: {N_u} x {N_v} = {total_pixels:,} pixels ({total_pixels/1e6:.2f} MP)")
print(f"Max sensor frame rate: {max_sensor_fps} fps")

# 5. Step 1 — Compute raw image size per pixel format

Fill in the image size (in bytes) for each of the three pixel formats in the table below.

| Format | Bit depth | Channels | Image size formula |
|---|---|---|---|
| Mono8 | 8 bits | 1 | $N_u \times N_v \times 8 / 8$ |
| Mono12 | 12 bits | 1 | $N_u \times N_v \times 12 / 8$ |
| RGB8 (Bayer demosaiced) | 8 bits | 3 | $N_u \times N_v \times 8 \times 3 / 8$ |


> [!NOTE]
> **RGB8 naming on the datasheet:** The Allied Vision Manta G-235C datasheet calls this
> format **RGB8Packed**. In this notebook "RGB8" always means a *fully demosaiced*
> colour image — the on-camera ISP has already interpolated the raw Bayer pattern into
> three separate 8-bit channels (R, G, B). The size formula is therefore
> $N_u \times N_v \times 3$ bytes: one byte per channel per pixel.
> Raw Bayer formats (BayerRG8 / BayerRG12) are listed in the datasheet but are not
> covered in this exercise.
> **Question 5.1:** Fill in the `???` below with the correct formulas.

In [ ]:
# --- Image size per format (bytes per frame) ---
# Replace ??? with your formula

size_mono8_bytes  = ???   # 8-bit grayscale
size_mono12_bytes = ???   # 12-bit grayscale (stored in 16-bit containers in practice)
size_rgb8_bytes   = ???   # 8-bit per channel, 3 channels

print("Image size per frame:")
print(f"  Mono8  : {size_mono8_bytes:>12,.0f} bytes  ({size_mono8_bytes/1e6:.2f} MB)")
print(f"  Mono12 : {size_mono12_bytes:>12,.0f} bytes  ({size_mono12_bytes/1e6:.2f} MB)")
print(f"  RGB8   : {size_rgb8_bytes:>12,.0f} bytes  ({size_rgb8_bytes/1e6:.2f} MB)")

# 6. Step 2 — Maximum transmittable frame rate

The bandwidth is 1 Gbps = $10^9$ bits per second.

$$
f_{\text{max,link}} = \frac{\text{bandwidth (bps)}}{\text{image size (bytes)} \times 8}
$$

But you also cannot exceed the sensor's hardware limit of 50.8 fps:

$$
f_{\text{max}} = \min\left(f_{\text{max,link}},\ f_{\text{sensor}\text{ max}}\right)
$$

> **Question 6.1:** Fill in the `???` below.

> **Question 6.2:** For which pixel format is the bandwidth the limiting factor? For which is the sensor hardware the limiting factor?

In [ ]:
# --- Maximum frame rate limited by bandwidth ---
# Replace ??? with your formula

fps_link_mono8  = ???   # bandwidth / (image_size_bytes * 8)
fps_link_mono12 = ???
fps_link_rgb8   = ???

# Effective maximum frame rate (cannot exceed sensor hardware limit)
fps_max_mono8  = min(fps_link_mono8,  max_sensor_fps)
fps_max_mono12 = min(fps_link_mono12, max_sensor_fps)
fps_max_rgb8   = min(fps_link_rgb8,   max_sensor_fps)

print("Maximum frame rate (bandwidth constraint):")
print(f"  Mono8  : {fps_link_mono8:>8.2f} fps  (link) → effective: {fps_max_mono8:.2f} fps")
print(f"  Mono12 : {fps_link_mono12:>8.2f} fps  (link) → effective: {fps_max_mono12:.2f} fps")
print(f"  RGB8   : {fps_link_rgb8:>8.2f} fps  (link) → effective: {fps_max_rgb8:.2f} fps")

# 7. Step 3 — Maximum recording time

With 10 GB of storage, recording at the effective maximum frame rate:

$$
t_{\text{max}} = \frac{\text{memory (bytes)}}{\text{image size (bytes)} \times f_{\text{max}}} \quad [\text{seconds}]
$$

> **Question 7.1:** Fill in the `???` below.

> **Question 7.2:** Convert the result to minutes. Is 10 GB a lot or a little for a robotic mission?

In [ ]:
# --- Maximum recording time ---
# Replace ??? with your formula

t_max_mono8_s  = ???   # seconds
t_max_mono12_s = ???
t_max_rgb8_s   = ???

print("Maximum recording time at effective max fps:")
print(f"  Mono8  : {t_max_mono8_s:>8.1f} s  = {t_max_mono8_s/60:.1f} min")
print(f"  Mono12 : {t_max_mono12_s:>8.1f} s  = {t_max_mono12_s/60:.1f} min")
print(f"  RGB8   : {t_max_rgb8_s:>8.1f} s  = {t_max_rgb8_s/60:.1f} min")

# 8. Sanity checks

Run this cell to verify your results.

In [ ]:
# --- Sanity checks ---

# Image sizes
assert abs(size_mono8_bytes  - 1936 * 1216 * 1) < 1,   "Mono8 size wrong"
assert abs(size_mono12_bytes - 1936 * 1216 * 1.5) < 1, "Mono12 size wrong"
assert abs(size_rgb8_bytes   - 1936 * 1216 * 3) < 1,   "RGB8 size wrong"
print(f"✓ Mono8  image size: {size_mono8_bytes/1e6:.3f} MB")
print(f"✓ Mono12 image size: {size_mono12_bytes/1e6:.3f} MB")
print(f"✓ RGB8   image size: {size_rgb8_bytes/1e6:.3f} MB")

# Frame rates: Mono8 at 1 Gbps should comfortably exceed sensor limit;
# RGB8 at 1 Gbps should be well below sensor limit.
assert fps_max_mono8 == max_sensor_fps, \
    f"Mono8 should be sensor-limited at {max_sensor_fps} fps"
assert fps_max_rgb8 < max_sensor_fps, \
    f"RGB8 should be bandwidth-limited (got {fps_max_rgb8:.2f} fps)"
print(f"\n✓ Mono8  effective fps: {fps_max_mono8:.2f} fps  (sensor-limited)")
print(f"✓ RGB8   effective fps: {fps_max_rgb8:.2f} fps  (bandwidth-limited)")

print("\nAll checks passed!")

# 9. Visualisation — bandwidth and storage comparison

In [ ]:
formats = ['Mono8', 'Mono12', 'RGB8']
image_sizes_mb = [
    size_mono8_bytes / 1e6,
    size_mono12_bytes / 1e6,
    size_rgb8_bytes / 1e6,
]
fps_values = [fps_max_mono8, fps_max_mono12, fps_max_rgb8]
record_times_min = [
    t_max_mono8_s / 60,
    t_max_mono12_s / 60,
    t_max_rgb8_s / 60,
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

colors = ['#4C72B0', '#DD8452', '#55A868']

axes[0].bar(formats, image_sizes_mb, color=colors)
axes[0].set_title('Image size per frame')
axes[0].set_ylabel('MB')
for i, v in enumerate(image_sizes_mb):
    axes[0].text(i, v + 0.05, f'{v:.2f}', ha='center', va='bottom', fontsize=10)

axes[1].bar(formats, fps_values, color=colors)
axes[1].axhline(max_sensor_fps, color='red', linestyle='--', linewidth=1.5,
                label=f'Sensor max: {max_sensor_fps} fps')
axes[1].set_title('Max effective frame rate')
axes[1].set_ylabel('fps')
axes[1].legend(fontsize=9)
for i, v in enumerate(fps_values):
    axes[1].text(i, v + 0.3, f'{v:.1f}', ha='center', va='bottom', fontsize=10)

axes[2].bar(formats, record_times_min, color=colors)
axes[2].set_title('Max recording time (10 GB)')
axes[2].set_ylabel('minutes')
for i, v in enumerate(record_times_min):
    axes[2].text(i, v + 0.05, f'{v:.1f} min', ha='center', va='bottom', fontsize=10)

plt.suptitle('Allied Vision Manta G-235C — bandwidth and storage analysis', fontsize=12)
plt.tight_layout()
plt.show()

# 10. Experiments

## Experiment A: Halve the resolution

Many cameras support **binning** or **decimation** which reduces the effective resolution by a factor of 2 in each dimension.

Change the resolution to 968 × 608 (half of each dimension) and recalculate.

Answer:
1. By what factor does the image size decrease?
2. By what factor does the maximum frame rate increase?
3. By what factor does the maximum recording time increase?
4. Is this always a good trade-off for a robotics application? When would you keep full resolution?

## Experiment B: Compression

JPEG compression typically achieves a 10:1 size reduction with acceptable quality loss.

Assume `size_compressed = size_raw / 10` and recalculate.

Answer:
1. How does the recording time change?
2. What are the trade-offs of using JPEG compression in a real-time robotics pipeline? (Hint: think about latency, CPU load, and lossy vs lossless)

## Experiment C: Design a mission

You are designing a 30-minute autonomous inspection mission for the MIRTE robot. The robot must continuously record full-resolution RGB8 images.

Answer:
1. How much storage is needed?
2. If storage is limited to 10 GB, what frame rate can you sustain for 30 minutes?
3. What alternative strategies (other than reducing frame rate) could help fit 30 minutes of footage into 10 GB?


In [ ]:
# --- Experiment C workspace ---
mission_duration_s = 30 * 60   # 30 minutes in seconds
memory_limit_bytes = 10e9

# Storage needed at max fps
storage_needed = size_rgb8_bytes * fps_max_rgb8 * mission_duration_s
print(f"Storage needed at {fps_max_rgb8:.2f} fps for 30 min: {storage_needed/1e9:.1f} GB")

# Sustainable fps within 10 GB
# Replace ??? with your formula
fps_sustainable = ???   # fps that keeps total storage ≤ 10 GB over 30 min
print(f"Sustainable frame rate within 10 GB: {fps_sustainable:.2f} fps")

# 11. Reflection questions

Answer these in a text cell or in your lab report.

1. The datasheet says the sensor can run at 50.8 fps. In your analysis, which pixel formats can actually achieve this rate over a 1 Gbps link? What upgrade to the hardware would allow all formats to run at sensor-maximum speed?

2. The calculation above uses *raw* (uncompressed) image sizes. In a real robot system, the ISP (Image Signal Processor) might apply Bayer demosaicing and JPEG encoding before transmission. How would this affect your bandwidth and storage calculations?

3. The MIRTE robot uses a rolling shutter sensor. If the robot is moving quickly, what artefacts might appear in the images? Would switching to a global shutter sensor change the bandwidth and storage requirements?

4. You want to add a second identical camera to the MIRTE robot for stereo depth estimation. Both cameras transmit over the same 1 Gbps link. What is the maximum frame rate for stereo RGB8 images? Is this sufficient for real-time robot navigation?


# References

- Lecture slides: *Practical Topics in Spatial AI*, Reza Sabzevari, TU Delft AE4ASM527
- Allied Vision Manta G-235 datasheet: https://www.alliedvision.com/en/camera-selector/detail/manta/g-235/
- Allied Vision bandwidth calculator: https://www.alliedvision.com/en/support/bandwidth-calculator/
